# Instrument Transposition Prototype

This notebook contains:
- Minimal instrument profiles (range/tessitura/polyphony)
- Two methods: **simple pitch correction (baseline)** vs **constraint-aware correction (advanced)**
- Soft-constraint reporting (tessitura) as metrics
- An evaluation harness to compare methods across instrument categories


In [27]:
import sys
!{sys.executable} -m pip install pandas numpy scipy mido ipykernel


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [28]:
# MIDI pitch numbers (C4 = 60)
INSTRUMENT_PROFILES = {
    # Woodwind
    "FLUTE": {
        "name": "Flute",
        "category": "woodwind",
        "min_pitch": 60,
        "max_pitch": 96,
        "preferred_low": 67,
        "preferred_high": 88,
        "max_polyphony": 1
    },
    "CLARINET": {
        "name": "Clarinet",
        "category": "woodwind",
        "min_pitch": 50,
        "max_pitch": 91,
        "preferred_low": 60,
        "preferred_high": 84,
        "max_polyphony": 1
    },

    # Vocal (example: Tenor)
    "TENOR_VOICE": {
        "name": "Tenor Voice",
        "category": "vocal",
        # Rough singing range; adjust as needed for your project
        "min_pitch": 48,   # C3
        "max_pitch": 69,   # A4
        "preferred_low": 52,  # E3
        "preferred_high": 64, # E4
        "max_polyphony": 1
    },

    # Brasswind (example: Trumpet)
    "TRUMPET": {
        "name": "Trumpet",
        "category": "brasswind",
        # Rough playable range; adjust as needed
        "min_pitch": 55,   # G3
        "max_pitch": 82,   # A#5
        "preferred_low": 60,  # C4
        "preferred_high": 76, # E5
        "max_polyphony": 1
    },

    # Strings (example: Violin)
    "VIOLIN": {
        "name": "Violin",
        "category": "strings",
        "min_pitch": 55,    # G3
        "max_pitch": 103,   # G7
        "preferred_low": 62,  # D4
        "preferred_high": 96, # C7
        "max_polyphony": 1
    },

    # Percussion (pitched) (example: Marimba)
    "MARIMBA": {
        "name": "Marimba",
        "category": "percussion",
        # Many marimbas are 4.3–5 octaves; this is a reasonable placeholder
        "min_pitch": 45,    # A2
        "max_pitch": 93,    # A6
        "preferred_low": 52,  # E3
        "preferred_high": 84, # C6
        "max_polyphony": 4   # mallet instruments can sound multiple notes
    },
}

# Convenience handles
FLUTE = INSTRUMENT_PROFILES["FLUTE"]
CLARINET = INSTRUMENT_PROFILES["CLARINET"]
TENOR_VOICE = INSTRUMENT_PROFILES["TENOR_VOICE"]
TRUMPET = INSTRUMENT_PROFILES["TRUMPET"]
VIOLIN = INSTRUMENT_PROFILES["VIOLIN"]
MARIMBA = INSTRUMENT_PROFILES["MARIMBA"]

## Core functions
Hard constraints: pitch range + polyphony. Soft metric: tessitura violations.


In [29]:
# Constraint functions
# Each note is represented as (pitch, start_time, duration, velocity)

from typing import List, Tuple, Dict, Any, Optional

# UPDATED: Added the 4th element (velocity) to the type hint
Note = Tuple[int, float, float, int]

def out_of_range(notes: List[Note], profile: Dict[str, Any]) -> bool:
    return any(
        (pitch < profile["min_pitch"]) or (pitch > profile["max_pitch"])
        for pitch, _, _, _ in notes # UPDATED: Unpack 4 elements
    )

def max_simultaneous_notes(notes: List[Note]) -> int:
    events = []
    for _, start, dur, _ in notes: # UPDATED: Unpack 4 elements
        events.append((start, +1))
        events.append((start + dur, -1))
    events.sort()
    current = 0
    max_poly = 0
    for _, delta in events:
        current += delta
        max_poly = max(max_poly, current)
    return max_poly

def tessitura_violations(notes: List[Note], profile: Dict[str, Any]) -> int:
    return sum(
        (pitch < profile["preferred_low"]) or (pitch > profile["preferred_high"])
        for pitch, _, _, _ in notes # UPDATED: Unpack 4 elements
    )

def shift_notes(notes: List[Note], semitones: int) -> List[Note]:
    # UPDATED: Unpack 4 elements and return 4 elements
    return [(pitch + semitones, start, dur, vel) for pitch, start, dur, vel in notes]

## Correction logic
We compare a simple baseline to a constraint-aware method that selects among feasible octave shifts by minimizing tessitura violations.


In [30]:
# Simple vs advanced (constraint aware) pitch correction

def simple_pitch_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simple baseline:
    - Only tries to make notes fit the absolute range via octave shifts.
    - Ignores polyphony and tessitura comfort.
    """
    if not out_of_range(notes, profile):
        return {
            "status": "ACCEPTED",
            "reason": "Baseline: already within range",
            "notes": notes,
            "shift": 0,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    # Try octave shifts -3..+3; pick the first range-feasible one (baseline behavior)
    for octave in range(-3, 4):
        shifted = shift_notes(notes, octave * 12)
        if not out_of_range(shifted, profile):
            return {
                "status": "CORRECTED",
                "reason": f"Baseline: first-fit octave shift {octave*12} semitones to satisfy range",
                "notes": shifted,
                "shift": octave * 12,
                "soft": {"tessitura_violations": tessitura_violations(shifted, profile)}
            }

    return {
        "status": "REJECTED",
        "reason": "Baseline: no octave shift can satisfy range",
        "notes": None,
        "shift": None,
        "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
    }


def find_best_feasible_transposition(notes: List[Note], profile: Dict[str, Any]) -> Tuple[Optional[List[Note]], Optional[int], Optional[int]]:
    """
    Advanced correction:
    - Enumerate all octave shifts that satisfy hard constraints: range + polyphony.
    - Choose the candidate that minimizes tessitura violations (soft metric),
      tie break by smallest absolute shift.
    Returns: (best_notes, best_shift, best_tessitura_violations)
    """
    candidates = []
    for octave in range(-3, 4):
        shift = octave * 12
        shifted = shift_notes(notes, shift)
        if out_of_range(shifted, profile):
            continue
        if max_simultaneous_notes(shifted) > profile["max_polyphony"]:
            continue
        tv = tessitura_violations(shifted, profile)
        candidates.append((tv, abs(shift), shift, shifted))

    if not candidates:
        return None, None, None

    candidates.sort(key=lambda x: (x[0], x[1]))  # min tessitura, then min |shift|
    tv, _, shift, best = candidates[0]
    return best, shift, tv


def constraint_aware_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Constraint aware method:
    - Hard constraint 1: polyphony (reject if violated, since we don't rewrite texture yet)
    - Hard constraint 2: range (correctable via octave shifts)
    - Soft signal: tessitura violations (reported, and used for selecting best shift)
    """
    poly = max_simultaneous_notes(notes)
    if poly > profile["max_polyphony"]:
        return {
            "status": "REJECTED",
            "reason": f"Hard constraint: polyphony {poly} exceeds limit {profile['max_polyphony']}",
            "notes": None,
            "shift": None,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    # If already within range, accept (but still report tessitura)
    if not out_of_range(notes, profile):
        tv = tessitura_violations(notes, profile)
        reason = "Hard constraints satisfied"
        if tv > 0:
            reason += f"; soft warning: {tv} tessitura violations"
        return {
            "status": "ACCEPTED",
            "reason": reason,
            "notes": notes,
            "shift": 0,
            "soft": {"tessitura_violations": tv}
        }

    # Range violated: try best feasible transposition (range+polyphony feasible, minimize tessitura)
    best, shift, tv = find_best_feasible_transposition(notes, profile)
    if best is None:
        return {
            "status": "REJECTED",
            "reason": "Hard constraint: no feasible octave shift satisfies range (and polyphony)",
            "notes": None,
            "shift": None,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    reason = f"Corrected: chose octave shift {shift} semitones (min tessitura violations)"
    if tv and tv > 0:
        reason += f"; soft warning: {tv} tessitura violations"
    return {
        "status": "CORRECTED",
        "reason": reason,
        "notes": best,
        "shift": shift,
        "soft": {"tessitura_violations": tv}
    }

In [31]:
def reduce_to_mono(notes: List[Note], strategy: str = "root_note") -> List[Note]:
    """
    Reduces a polyphonic sequence of notes to a strict monophonic line.
    Handles the 4-element tuple: (pitch, start, dur, vel)
    """
    if not notes:
        return []

    # FIX: Sort based on the selected strategy to avoid the screaming violin bias
    if strategy == "top_note":
        # Highest pitch first
        sorted_notes = sorted(notes, key=lambda x: (x[1], -x[0]))
    else:
        # Lowest pitch (root) first for a stronger fundamental
        sorted_notes = sorted(notes, key=lambda x: (x[1], x[0]))

    mono_notes = []
    current_start = -1.0
    
    # Step 1: Extract the selected note for each onset time
    top_notes = []
    for pitch, start, dur, vel in sorted_notes:
        # Group by exact (or very close) start time to handle chords
        if abs(start - current_start) > 0.001:
            top_notes.append([pitch, start, dur, vel])
            current_start = start
            
    # Step 2: Fix durations to ensure max_polyphony == 1 (no legato overlap bleed)
    for i in range(len(top_notes)):
        pitch, start, dur, vel = top_notes[i]
        
        if i < len(top_notes) - 1:
            next_start = top_notes[i+1][1]
            # If the current note bleeds into the next note, truncate it
            if start + dur > next_start:
                dur = next_start - start
        
        # Only keep notes that still have a positive duration
        if dur > 0:
            mono_notes.append((pitch, float(start), float(dur), vel))
            
    return mono_notes


def texture_adapted_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Phase 2 pipeline: Adapts texture before applying constraints.
    Input -> Reduce Texture (if needed) -> Transpose -> Constraint Check
    """
    adapted_notes = notes
    
    # If the target is monophonic and the input has chords/overlaps
    if profile["max_polyphony"] == 1 and max_simultaneous_notes(notes) > 1:
        # FIX: Tell the reducer to grab the root note instead of the top note
        adapted_notes = reduce_to_mono(notes, strategy="root_note")
        
    # Pass the (potentially adapted) notes into your existing advanced logic
    result = constraint_aware_correction(adapted_notes, profile)
    
    # Update the reason string so it shows up in your dataframe!
    if adapted_notes != notes:
        if result["status"] in ["ACCEPTED", "CORRECTED"]:
            result["reason"] = "Texture reduced to mono; " + result["reason"]
        else:
            result["reason"] = "Texture reduced to mono, BUT " + result["reason"]
            
    return result

## Quick test cases


In [32]:
# Flute tests (Updated with velocity = 80)
melody_out_of_range = [(48, 0.0, 0.5, 80), (50, 0.5, 0.5, 80), (52, 1.0, 0.5, 80)]
chord = [(72, 0.0, 1.0, 80), (76, 0.0, 1.0, 80), (79, 0.0, 1.0, 80)]
valid_melody = [(72, 0.0, 0.5, 80), (74, 0.5, 0.5, 80), (76, 1.0, 0.5, 80)]
boundary_melody = [(FLUTE["min_pitch"], 0.0, 0.5, 80), (FLUTE["max_pitch"], 0.5, 0.5, 80)]
unfixable_melody = [(40, 0.0, 0.5, 80), (100, 0.5, 0.5, 80)]

# Clarinet tests
clarinet_valid = [(62, 0.0, 0.5, 80), (65, 0.5, 0.5, 80), (69, 1.0, 0.5, 80)]
clarinet_out_of_range = [(45, 0.0, 0.5, 80), (47, 0.5, 0.5, 80), (49, 1.0, 0.5, 80)]

# Cross-category sanity: Marimba chord should be acceptable (polyphonic)
marimba_chord = [(60, 0.0, 1.0, 80), (64, 0.0, 1.0, 80), (67, 0.0, 1.0, 80)]

print("Baseline (Flute chord):", simple_pitch_correction(chord, FLUTE))
print("Constraint aware (Flute chord):", constraint_aware_correction(chord, FLUTE))

# NEW: Test the texture adaptation!
print("Texture adapted (Flute chord):", texture_adapted_correction(chord, FLUTE))

print("Constraint aware (Marimba chord):", constraint_aware_correction(marimba_chord, MARIMBA))

Baseline (Flute chord): {'status': 'ACCEPTED', 'reason': 'Baseline: already within range', 'notes': [(72, 0.0, 1.0, 80), (76, 0.0, 1.0, 80), (79, 0.0, 1.0, 80)], 'shift': 0, 'soft': {'tessitura_violations': 0}}
Constraint aware (Flute chord): {'status': 'REJECTED', 'reason': 'Hard constraint: polyphony 3 exceeds limit 1', 'notes': None, 'shift': None, 'soft': {'tessitura_violations': 0}}
Texture adapted (Flute chord): {'status': 'ACCEPTED', 'reason': 'Texture reduced to mono; Hard constraints satisfied', 'notes': [(72, 0.0, 1.0, 80)], 'shift': 0, 'soft': {'tessitura_violations': 0}}
Constraint aware (Marimba chord): {'status': 'ACCEPTED', 'reason': 'Hard constraints satisfied', 'notes': [(60, 0.0, 1.0, 80), (64, 0.0, 1.0, 80), (67, 0.0, 1.0, 80)], 'shift': 0, 'soft': {'tessitura_violations': 0}}


## Evaluation harness
Runs the same clips across instruments and methods and returns a results table.


In [33]:
import pandas as pd

# UPDATED: Added the texture adaptation baseline!
METHODS = {
    "simple_pitch": simple_pitch_correction,
    "constraint_aware": constraint_aware_correction,
    "texture_adapted": texture_adapted_correction,
}

EVAL_CLIPS = {
    "flute_out_of_range": melody_out_of_range,
    "flute_chord": chord,
    "flute_valid": valid_melody,
    "flute_boundary": boundary_melody,
    "flute_unfixable": unfixable_melody,
    "clarinet_valid": clarinet_valid,
    "clarinet_out_of_range": clarinet_out_of_range,
    "marimba_chord": marimba_chord,
}

EVAL_INSTRUMENTS = {
    "FLUTE": FLUTE,
    "CLARINET": CLARINET,
    "TENOR_VOICE": TENOR_VOICE,
    "TRUMPET": TRUMPET,
    "VIOLIN": VIOLIN,
    "MARIMBA": MARIMBA,
}

def run_harness(clips, instruments, methods) -> pd.DataFrame:
    rows = []
    for clip_name, notes in clips.items():
        for inst_key, profile in instruments.items():
            for method_name, fn in methods.items():
                res = fn(notes, profile)
                playable = (res["notes"] is not None)
                rows.append({
                    "clip": clip_name,
                    "instrument": inst_key,
                    "category": profile.get("category", ""),
                    "method": method_name,
                    "status": res.get("status"),
                    "playable": playable,
                    "shift": res.get("shift"),
                    "tessitura_violations": (res.get("soft", {}) or {}).get("tessitura_violations"),
                    "reason": res.get("reason"),
                })
    return pd.DataFrame(rows)

df_results = run_harness(EVAL_CLIPS, EVAL_INSTRUMENTS, METHODS)

df_results.sort_values(["clip", "instrument", "method"]).head(30)

,clip,instrument,category,method,status,playable,shift,tessitura_violations,reason
112,clarinet_out_of_range,CLARINET,woodwind,constraint_aware,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
111,clarinet_out_of_range,CLARINET,woodwind,simple_pitch,CORRECTED,True,12.0,2,Baseline: first-fit octave shift 12 semitones ...
113,clarinet_out_of_range,CLARINET,woodwind,texture_adapted,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
109,clarinet_out_of_range,FLUTE,woodwind,constraint_aware,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
108,clarinet_out_of_range,FLUTE,woodwind,simple_pitch,CORRECTED,True,24.0,0,Baseline: first-fit octave shift 24 semitones ...
110,clarinet_out_of_range,FLUTE,woodwind,texture_adapted,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
124,clarinet_out_of_range,MARIMBA,percussion,constraint_aware,ACCEPTED,True,0.0,3,Hard constraints satisfied; soft warning: 3 te...
123,clarinet_out_of_range,MARIMBA,percussion,simple_pitch,ACCEPTED,True,0.0,3,Baseline: already within range
125,clarinet_out_of_range,MARIMBA,percussion,texture_adapted,ACCEPTED,True,0.0,3,Hard constraints satisfied; soft warning: 3 te...
115,clarinet_out_of_range,TENOR_VOICE,vocal,constraint_aware,CORRECTED,True,12.0,0,Corrected: chose octave shift 12 semitones (mi...


## Metrics used in the evaluation harness

For each *(clip × instrument × method)* run, the harness reports:

**Hard-constraint metrics (playability):**
- `hard_feasible`: output satisfies **range** and **polyphony** constraints for the target instrument.
- `status`: `ACCEPTED`, `CORRECTED`, or `REJECTED` (system behavior).

**Soft-constraint metrics (comfort):**
- `tessitura_violations`: number of notes outside the preferred register.

**Change magnitude:**
- `shift`: semitone shift applied (global octave shifts in this prototype).
- `abs_shift`: `|shift|` (how far the music moved).

We also summarize feasibility and tessitura per instrument and method.

## Representative instruments per category (used in this notebook)

- **vocal:** Tenor Voice (`TENOR_VOICE`)
- **woodwind:** Flute (`FLUTE`), Clarinet (`CLARINET`)
- **brasswind:** Trumpet (`TRUMPET`)
- **strings:** Violin (`VIOLIN`)
- **percussion (pitched):** Marimba (`MARIMBA`)

In [34]:
# hard feasibility metric: check returned notes against hard constraints

def hard_feasible(notes_out, profile):
    """True playability: range + polyphony must be satisfied."""
    if notes_out is None:
        return False
    if out_of_range(notes_out, profile):
        return False
    if max_simultaneous_notes(notes_out) > profile["max_polyphony"]:
        return False
    return True

# Re run harness results with a clean hard_feasible flag
df_results2 = df_results.copy()

# Reconstruct output notes from the stored shift (and apply texture reduction if needed)
def reconstruct_output_notes(row):
    base_notes = EVAL_CLIPS[row["clip"]]
    shift = row["shift"]
    
    if shift is None:
        return None
    try:
        shift = int(shift)
    except Exception:
        return None
        
    # NEW: If the method adapted the texture, we must apply the same reduction 
    # here before shifting so the hard_feasible check doesn't falsely fail!
    if row["method"] == "texture_adapted":
        profile = EVAL_INSTRUMENTS[row["instrument"]]
        if profile["max_polyphony"] == 1 and max_simultaneous_notes(base_notes) > 1:
            base_notes = reduce_to_mono(base_notes, strategy="top_note")
            
    return shift_notes(base_notes, shift)

df_results2["output_notes"] = df_results2.apply(reconstruct_output_notes, axis=1)
df_results2["hard_feasible"] = df_results2.apply(lambda r: hard_feasible(r["output_notes"], EVAL_INSTRUMENTS[r["instrument"]]), axis=1)
df_results2["abs_shift"] = df_results2["shift"].abs()

wide_hard = df_results2.pivot_table(
    index=["clip","instrument","category"],
    columns="method",
    values=["hard_feasible","tessitura_violations","abs_shift","status"],
    aggfunc="first"
)
wide_hard.columns = [f"{a}_{b}" for a,b in wide_hard.columns]
wide_hard = wide_hard.reset_index()

wide_hard.head(20)

,clip,instrument,category,abs_shift_constraint_aware,abs_shift_simple_pitch,abs_shift_texture_adapted,hard_feasible_constraint_aware,hard_feasible_simple_pitch,hard_feasible_texture_adapted,status_constraint_aware,status_simple_pitch,status_texture_adapted,tessitura_violations_constraint_aware,tessitura_violations_simple_pitch,tessitura_violations_texture_adapted
0,clarinet_out_of_range,CLARINET,woodwind,24.0,12.0,24.0,True,True,True,CORRECTED,CORRECTED,CORRECTED,0,2,0
1,clarinet_out_of_range,FLUTE,woodwind,24.0,24.0,24.0,True,True,True,CORRECTED,CORRECTED,CORRECTED,0,0,0
2,clarinet_out_of_range,MARIMBA,percussion,0.0,0.0,0.0,True,True,True,ACCEPTED,ACCEPTED,ACCEPTED,3,3,3
3,clarinet_out_of_range,TENOR_VOICE,vocal,12.0,12.0,12.0,True,True,True,CORRECTED,CORRECTED,CORRECTED,0,0,0
4,clarinet_out_of_range,TRUMPET,brasswind,24.0,12.0,24.0,True,True,True,CORRECTED,CORRECTED,CORRECTED,0,2,0
5,clarinet_out_of_range,VIOLIN,strings,24.0,12.0,24.0,True,True,True,CORRECTED,CORRECTED,CORRECTED,0,3,0
6,clarinet_valid,CLARINET,woodwind,0.0,0.0,0.0,True,True,True,ACCEPTED,ACCEPTED,ACCEPTED,0,0,0
7,clarinet_valid,FLUTE,woodwind,0.0,0.0,0.0,True,True,True,ACCEPTED,ACCEPTED,ACCEPTED,2,2,2
8,clarinet_valid,MARIMBA,percussion,0.0,0.0,0.0,True,True,True,ACCEPTED,ACCEPTED,ACCEPTED,0,0,0
9,clarinet_valid,TENOR_VOICE,vocal,0.0,0.0,0.0,True,True,True,ACCEPTED,ACCEPTED,ACCEPTED,2,2,2


In [35]:
import numpy as np

summary = (
    df_results2.groupby(["instrument","category","method"])
    .agg(
        hard_feasible_rate=("hard_feasible","mean"),
        avg_tessitura=("tessitura_violations","mean"),
        avg_abs_shift=("abs_shift","mean"),
        accepted=("status", lambda s: np.mean(s=="ACCEPTED")),
        corrected=("status", lambda s: np.mean(s=="CORRECTED")),
        rejected=("status", lambda s: np.mean(s=="REJECTED")),
        n=("status","count"),
    )
    .reset_index()
    .sort_values(["category","instrument","method"])
)

summary

,instrument,category,method,hard_feasible_rate,avg_tessitura,avg_abs_shift,accepted,corrected,rejected,n
12,TRUMPET,brasswind,constraint_aware,0.500,0.500,9.000000,0.250,0.250,0.500,8
13,TRUMPET,brasswind,simple_pitch,0.500,0.750,4.000000,0.500,0.250,0.250,8
14,TRUMPET,brasswind,texture_adapted,0.750,0.375,6.000000,0.500,0.250,0.250,8
6,MARIMBA,percussion,constraint_aware,0.875,1.000,1.714286,0.750,0.125,0.125,8
7,MARIMBA,percussion,simple_pitch,0.875,1.000,1.714286,0.750,0.125,0.125,8
8,MARIMBA,percussion,texture_adapted,0.875,1.000,1.714286,0.750,0.125,0.125,8
15,VIOLIN,strings,constraint_aware,0.625,0.500,9.600000,0.375,0.250,0.375,8
16,VIOLIN,strings,simple_pitch,0.625,1.000,3.428571,0.625,0.250,0.125,8
17,VIOLIN,strings,texture_adapted,0.875,0.500,6.857143,0.625,0.250,0.125,8
9,TENOR_VOICE,vocal,constraint_aware,0.500,1.375,6.000000,0.250,0.250,0.500,8


In [36]:
# Statistical tests to check if advanced is working

import numpy as np
import scipy.stats as stats

def mcnemar_exact(a_bool, b_bool):
    a = np.asarray(a_bool, dtype=bool)  
    b = np.asarray(b_bool, dtype=bool)  
    n01 = np.sum((~a) & (b))  
    n10 = np.sum((a) & (~b))  
    n00 = np.sum((~a) & (~b))
    n11 = np.sum((a) & (b))
    n = n01 + n10
    p = 2 * stats.binomtest(min(n01, n10), n=n, p=0.5).pvalue if n > 0 else 1.0
    return n00, n01, n10, n11, p

print("=== PHASE 1: Constraint Aware vs Simple Pitch ===")
# 1) Hard feasibility: McNemar test
n00, n01, n10, n11, p_mcnemar = mcnemar_exact(
    wide_hard["hard_feasible_constraint_aware"],
    wide_hard["hard_feasible_simple_pitch"],
)
print("McNemar contingency [n00 n01 n10 n11]:", [n00, n01, n10, n11])
print("McNemar exact p-value:", p_mcnemar)

# 2) Soft metric: tessitura violations (paired)
both_feasible_p1 = wide_hard[(wide_hard["hard_feasible_constraint_aware"]==True) & (wide_hard["hard_feasible_simple_pitch"]==True)].copy()
print("Num paired cases where both methods are hard-feasible:", len(both_feasible_p1))

if len(both_feasible_p1) > 1:
    delta_tess = both_feasible_p1["tessitura_violations_constraint_aware"] - both_feasible_p1["tessitura_violations_simple_pitch"]
    t_res = stats.ttest_rel(both_feasible_p1["tessitura_violations_constraint_aware"], both_feasible_p1["tessitura_violations_simple_pitch"])
    print("Mean delta tessitura (advanced - simple):", delta_tess.mean())
    print("Paired t-test p (tessitura):", t_res.pvalue)


print("\n=== PHASE 2: Texture Adapted vs Constraint Aware ===")
# 1) Hard feasibility: McNemar test
n00_t, n01_t, n10_t, n11_t, p_mcnemar_t = mcnemar_exact(
    wide_hard["hard_feasible_texture_adapted"],
    wide_hard["hard_feasible_constraint_aware"],
)
print("McNemar contingency [n00 n01 n10 n11]:", [n00_t, n01_t, n10_t, n11_t])
print("McNemar exact p-value:", p_mcnemar_t)

# 2) Soft metric: tessitura violations (paired)
both_feasible_p2 = wide_hard[(wide_hard["hard_feasible_texture_adapted"]==True) & (wide_hard["hard_feasible_constraint_aware"]==True)].copy()
print("Num paired cases where both methods are hard-feasible:", len(both_feasible_p2))

if len(both_feasible_p2) > 1:
    delta_tess_t = both_feasible_p2["tessitura_violations_texture_adapted"] - both_feasible_p2["tessitura_violations_constraint_aware"]
    t_res_t = stats.ttest_rel(both_feasible_p2["tessitura_violations_texture_adapted"], both_feasible_p2["tessitura_violations_constraint_aware"])
    print("Mean delta tessitura (texture adapted - constraint aware):", delta_tess_t.mean())
    print("Paired t-test p (tessitura):", t_res_t.pvalue)

=== PHASE 1: Constraint Aware vs Simple Pitch ===
McNemar contingency [n00 n01 n10 n11]: [np.int64(19), np.int64(0), np.int64(0), np.int64(29)]
McNemar exact p-value: 1.0
Num paired cases where both methods are hard-feasible: 29
Mean delta tessitura (advanced - simple): -0.4482758620689655
Paired t-test p (tessitura): 0.016684047108580662

=== PHASE 2: Texture Adapted vs Constraint Aware ===
McNemar contingency [n00 n01 n10 n11]: [np.int64(9), np.int64(0), np.int64(10), np.int64(29)]
McNemar exact p-value: 0.00390625
Num paired cases where both methods are hard-feasible: 29
Mean delta tessitura (texture adapted - constraint aware): 0.0
Paired t-test p (tessitura): nan


**Hard feasibility (playability):**  
Both methods produced identical feasibility outcomes (McNemar p = 1.0).  
This shows the advanced method does not create playable results where none existed; instead, it selects among already playable mappings.

**Comfort (tessitura):**  
The constraint aware method significantly reduced tessitura violations (paired t-test p = 0.0167, Wilcoxon p = 0.0256).  
This indicates improved ergonomic playability through better register selection.

**Magnitude of change:**  
The advanced method did not significantly increase pitch movement (|shift| p = 0.10), suggesting improvements are due to smarter octave choice rather than excessive modification.

**Conclusion:**  
The constraint aware correction functions as a refinement mechanism rather than a full repair. It preserves feasibility while improving the comfort of the resulting music.

In [37]:
# Add GM MIDI program numbers to profiles (0-127) for export/listening
# General MIDI programs (1-128). MIDI messages use 0-127.
# Flute=74->73, Clarinet=72->71, Trumpet=57->56, Violin=41->40, Marimba=13->12, Choir Aahs=53->52

INSTRUMENT_PROFILES["FLUTE"]["midi_program"] = 73
INSTRUMENT_PROFILES["CLARINET"]["midi_program"] = 71
INSTRUMENT_PROFILES["TRUMPET"]["midi_program"] = 56
INSTRUMENT_PROFILES["VIOLIN"]["midi_program"] = 40
INSTRUMENT_PROFILES["MARIMBA"]["midi_program"] = 12
INSTRUMENT_PROFILES["TENOR_VOICE"]["midi_program"] = 52

In [38]:
pip install mido


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
from typing import List, Tuple, Dict, Any
import mido
from collections import defaultdict

# New Note Definition: (pitch, start_beats, dur_beats, velocity)
Note = Tuple[int, float, float, int]

# UPDATED: Added 'int' to the return type hint for the ticks_per_beat
def midi_to_tracks(path: str) -> Tuple[List[Dict[str, Any]], float, int]:
    """
    Reads a MIDI file and extracts notes per track.
    Handles CC64 (sustain pedal) and captures initial tempo.
    Returns: (list_of_track_dicts, bpm, ticks_per_beat)
    """
    mid = mido.MidiFile(path)
    tpb = mid.ticks_per_beat
    
    # Default tempo (120 BPM = 500,000 microseconds per beat)
    initial_tempo = 500000 
    
    tracks_data = []
    
    for i, track in enumerate(mid.tracks):
        time_ticks = 0
        active_notes = defaultdict(list)
        pedal_sustained = defaultdict(list)
        pedal_down = False
        
        notes = []
        program = 0
        channel = None
        track_name = f"Track {i}"
        
        for msg in track:
            time_ticks += msg.time
            
            if msg.type == 'set_tempo' and initial_tempo == 500000:
                # Capture the first tempo change as the global BPM
                initial_tempo = msg.tempo
                
            elif msg.type == 'track_name':
                track_name = msg.name
                
            elif msg.type == 'program_change':
                program = msg.program
                if channel is None:
                    channel = msg.channel
                    
            elif msg.type == 'control_change' and msg.control == 64:
                # Sustain pedal logic
                if msg.value >= 64:
                    pedal_down = True
                else:
                    pedal_down = False
                    # Release all notes waiting on the pedal
                    t_beats = time_ticks / tpb
                    for p, starts in pedal_sustained.items():
                        for (start, vel) in starts:
                            dur = max(0.0, t_beats - start)
                            notes.append((p, float(start), float(dur), vel))
                    pedal_sustained.clear()
                    
            elif msg.type in ('note_on', 'note_off'):
                if channel is None:
                    channel = getattr(msg, 'channel', 0)
                    
                pitch = msg.note
                t_beats = time_ticks / tpb
                vel = getattr(msg, 'velocity', 0)
                is_note_off = (msg.type == 'note_off') or (vel == 0)
                
                if not is_note_off:
                    active_notes[pitch].append((t_beats, vel))
                else:
                    if active_notes[pitch]:
                        start, onset_vel = active_notes[pitch].pop(0)
                        if pedal_down:
                            # Wait for pedal to lift before ending duration
                            pedal_sustained[pitch].append((start, onset_vel))
                        else:
                            dur = max(0.0, t_beats - start)
                            notes.append((pitch, float(start), float(dur), onset_vel))
        
        # Flush any remaining notes at the end of the track
        t_beats = time_ticks / tpb
        for p, starts in active_notes.items():
            for (start, vel) in starts:
                notes.append((p, float(start), max(0.0, t_beats - start), vel))
        for p, starts in pedal_sustained.items():
            for (start, vel) in starts:
                notes.append((p, float(start), max(0.0, t_beats - start), vel))
                
        if notes: # Only keep tracks that actually contain notes
            notes.sort(key=lambda x: (x[1], x[0]))
            tracks_data.append({
                "track_idx": i,
                "name": track_name,
                "channel": channel if channel is not None else 0,
                "program": program,
                "notes": notes
            })
            
    bpm = mido.tempo2bpm(initial_tempo)
    
    # UPDATED: Return the ticks_per_beat alongside the data and bpm
    return tracks_data, bpm, tpb


def notes_to_midi(notes, out_path, program=0, bpm=120, ticks_per_beat=480):
    """
    Write notes to a MIDI file, injecting generative CC automation (Expression/Dynamics) 
    to make sustained instruments sound human and realistic.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    mid = mido.MidiFile(ticks_per_beat=ticks_per_beat)
    track = mido.MidiTrack()
    mid.tracks.append(track)

    tempo = mido.bpm2tempo(int(bpm)) 
    track.append(mido.MetaMessage("set_tempo", tempo=tempo, time=0))
    track.append(mido.Message("program_change", program=int(program), time=0))

    events = []
    
    for pitch, start, dur, vel in notes:
        start_tick = int(round(start * ticks_per_beat))
        end_tick = int(round((start + dur) * ticks_per_beat))
        end_tick = max(end_tick, start_tick + 1)
        
        # 1. Add the core Note On / Note Off events
        events.append((start_tick, "on", int(pitch), int(vel)))
        events.append((end_tick, "off", int(pitch), 0))

        # 2. GENERATIVE EXPRESSION LAYER
        # If the note is held for more than half a beat, give it a human "swell"
        if dur > 0.5:
            num_steps = 16 # Resolution of the automation curve
            step_ticks = (end_tick - start_tick) // num_steps
            
            for i in range(num_steps):
                cc_tick = start_tick + (i * step_ticks)
                progress = i / (num_steps - 1)
                
                # Create an arc: Start at 40, swell to 100 in the middle, fade to 60 at the end
                if progress < 0.5:
                    cc_val = int(40 + (progress * 2 * 60)) 
                else:
                    cc_val = int(100 - ((progress - 0.5) * 2 * 40)) 
                    
                # Inject CC 11 (Expression) and CC 1 (Mod Wheel / Dynamics)
                events.append((cc_tick, "cc", 11, cc_val))
                events.append((cc_tick, "cc", 1, cc_val))

    # Sort events: Time first. If tied: note_off (0) < cc (1) < note_on (2)
    # This prevents notes from being cut off early if a CC happens at the exact same tick
    events.sort(key=lambda e: (e[0], 0 if e[1] == "off" else (1 if e[1] == "cc" else 2)))

    # Write the sorted events to the track
    last_tick = 0
    for event in events:
        tick = event[0]
        typ = event[1]
        delta = tick - last_tick
        last_tick = tick
        
        if typ == "on":
            track.append(mido.Message("note_on", note=event[2], velocity=event[3], time=delta))
        elif typ == "off":
            track.append(mido.Message("note_off", note=event[2], velocity=0, time=delta))
        elif typ == "cc":
            # event[2] is CC number, event[3] is CC value
            track.append(mido.Message("control_change", control=event[2], value=event[3], time=delta))

    track.append(mido.MetaMessage("end_of_track", time=0))
    mid.save(str(out_path))
    return str(out_path)


def assign_track_roles(tracks_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Analyzes track metadata and notes to assign 'melody' or 'accompaniment' roles.
    """
    if not tracks_data:
        return tracks_data
        
    def calculate_melody_score(notes: List[Note]) -> float:
        if not notes:
            return -1000.0
        avg_pitch = sum(n[0] for n in notes) / len(notes)
        overlaps = 0
        active_ends = []
        for n in notes:
            pitch, start, dur, vel = n
            active_ends = [e for e in active_ends if e <= start + 0.05]
            overlaps += len(active_ends)
            active_ends.append(start + dur)
        monophony_ratio = 1.0 - min(1.0, overlaps / len(notes))
        return avg_pitch + (monophony_ratio * 30)

    best_score = -float('inf')
    melody_idx = 0
    for i, track in enumerate(tracks_data):
        if track["channel"] == 9: # Skip drums
            continue
        score = calculate_melody_score(track["notes"])
        if score > best_score:
            best_score = score
            melody_idx = i
            
    for i, track in enumerate(tracks_data):
        track["role"] = "melody" if i == melody_idx else "accompaniment"
        
    return tracks_data


def patch_only_baseline(in_path, out_path, program=0):
    """
    Baseline: keep the MIDI the same, but change the instrument patch.
    """
    in_path = Path(in_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    mid = mido.MidiFile(str(in_path))
    for track in mid.tracks:
        new_msgs = []
        inserted = False
        for msg in track:
            if msg.type == "program_change":
                continue
            if not inserted and not msg.is_meta:
                new_msgs.append(mido.Message("program_change", program=int(program), time=0))
                inserted = True
            new_msgs.append(msg)
        if not inserted:
            new_msgs.insert(0, mido.Message("program_change", program=int(program), time=0))
        track[:] = new_msgs

    mid.save(str(out_path))
    return str(out_path)

In [40]:

import pandas as pd
from pathlib import Path
import shutil

FILE_METHODS = {
    "patch_only": None,
    "simple_pitch": simple_pitch_correction,
    "constraint_aware": constraint_aware_correction,
    "texture_adapted": texture_adapted_correction
}

def get_melody_preservation(input_notes, output_notes, shift):
    """
    Proxy for melody preservation:
    1. Un-shifts the output notes back to original pitch.
    2. Reduces both input and output to monophonic top-lines.
    3. Calculates the overlap percentage of (pitch, start_time) pairs.
    """
    if not output_notes or not input_notes:
        return 0.0

    shift = shift or 0
    unshifted_out = [(p - shift, s, d, v) for p, s, d, v in output_notes]

    in_mono = reduce_to_mono(input_notes)
    out_mono = reduce_to_mono(unshifted_out)

    # Use rounding to avoid floating point precision issues
    in_set = {(p, round(s, 3)) for p, s, _, _ in in_mono}
    out_set = {(p, round(s, 3)) for p, s, _, _ in out_mono}

    if not in_set:
        return 1.0

    overlap = len(in_set.intersection(out_set))
    return overlap / len(in_set)


def export_eval_clips_to_dataset(clips, out_dir, bpm=120, program=0):
    """Write the hardcoded symbolic examples to MIDI so they can join the file harness."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    written = []
    for clip_name, notes in clips.items():
        out_path = out_dir / f"{clip_name}.mid"
        notes_to_midi(notes, out_path, program=program, bpm=bpm)
        written.append(out_path)
    return written


def copy_midi_tree(src_dir, dst_dir):
    """Copy a MIDI dataset while preserving its relative folder layout."""
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)

    copied = 0
    for midi_path in sorted(p for p in src_dir.glob("**/*") if p.suffix.lower() in (".mid", ".midi")):
        rel_path = midi_path.relative_to(src_dir)
        out_path = dst_dir / rel_path
        out_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(midi_path, out_path)
        copied += 1
    return copied


def build_aggregated_dataset(eval_clips, aggregated_dir="datasets/inputs/aggregated", external_sources=None):
    """
    Combine the hardcoded notebook examples with any external MIDI corpora.
    Each source gets its own top-level folder so downstream metrics can stay source-aware.
    """
    aggregated_dir = Path(aggregated_dir)
    if aggregated_dir.exists():
        shutil.rmtree(aggregated_dir)
    aggregated_dir.mkdir(parents=True, exist_ok=True)

    summary = {}
    hardcoded_dir = aggregated_dir / "hardcoded_eval"
    summary["hardcoded_eval"] = len(export_eval_clips_to_dataset(eval_clips, hardcoded_dir))

    for source_name, source_dir in (external_sources or {}).items():
        source_dir = Path(source_dir)
        if not source_dir.exists():
            continue
        summary[source_name] = copy_midi_tree(source_dir, aggregated_dir / source_name)

    return aggregated_dir, summary


def pick_demo_triplet(results_df, dataset_root, instrument="VIOLIN", preferred_clip=None, preferred_sources=None, min_notes_dropped=None):
    """Pick one original/naive/pipeline trio from the aggregated results table."""
    candidates = results_df[results_df["instrument"] == instrument].copy()

    if preferred_sources:
        candidates = candidates[candidates["dataset_source"].isin(preferred_sources)]
    if preferred_clip is not None:
        candidates = candidates[candidates["clip"] == preferred_clip]
    if min_notes_dropped is not None:
        candidates = candidates[candidates["notes_dropped"] >= min_notes_dropped]

    if candidates.empty:
        return None

    ranked = (
        candidates.groupby(["dataset_source", "relative_file", "clip"], as_index=False)
        .agg(
            max_notes_dropped=("notes_dropped", "max"),
            max_input_polyphony=("input_max_polyphony", "max"),
        )
        .sort_values(["max_notes_dropped", "max_input_polyphony", "relative_file"], ascending=[False, False, True])
    )

    for _, row in ranked.iterrows():
        triplet_rows = candidates[
            (candidates["dataset_source"] == row["dataset_source"]) &
            (candidates["relative_file"] == row["relative_file"]) &
            (candidates["clip"] == row["clip"])
        ]
        methods = set(triplet_rows["method"])
        if not {"patch_only", "texture_adapted"}.issubset(methods):
            continue

        patch_row = triplet_rows[triplet_rows["method"] == "patch_only"].iloc[0]
        texture_row = triplet_rows[triplet_rows["method"] == "texture_adapted"].iloc[0]

        return {
            "dataset_source": row["dataset_source"],
            "relative_file": row["relative_file"],
            "clip": row["clip"],
            "max_notes_dropped": row["max_notes_dropped"],
            "original_file": Path(dataset_root) / row["relative_file"],
            "patch_only_file": Path(patch_row["output_midi"]),
            "texture_adapted_file": Path(texture_row["output_midi"]),
        }

    return None


def run_file_harness(dataset_dir, instruments, methods=FILE_METHODS, outputs_dir="outputs"):
    """
    Process a folder of MIDI files using track-aware parsing and texture metrics.
    The harness keeps track of which aggregated source each file came from.
    """
    dataset_dir = Path(dataset_dir)
    outputs_dir = Path(outputs_dir)
    outputs_dir.mkdir(parents=True, exist_ok=True)

    midi_files = sorted([p for p in dataset_dir.glob("**/*") if p.suffix.lower() in (".mid", ".midi")])
    if not midi_files:
        raise FileNotFoundError(f"No .mid/.midi files found under: {dataset_dir}")

    rows = []
    for midi_path in midi_files:
        relative_file = midi_path.relative_to(dataset_dir)
        clip_name = midi_path.stem
        clip_id = "__".join(relative_file.with_suffix("").parts)
        dataset_source = relative_file.parts[0] if len(relative_file.parts) > 1 else dataset_dir.name

        if len(relative_file.parts) > 1:
            output_relative = Path(*relative_file.parts[1:])
        else:
            output_relative = Path(relative_file.name)

        tracks_data, parsed_bpm, parsed_tpb = midi_to_tracks(str(midi_path))
        tracks_data = assign_track_roles(tracks_data)

        base_notes = []
        for track in tracks_data:
            if track.get("role") == "melody":
                base_notes = track["notes"]
                break
        if not base_notes and tracks_data:
            base_notes = tracks_data[0]["notes"]

        in_poly = max_simultaneous_notes(base_notes) if base_notes else 0

        for inst_key, profile in instruments.items():
            program = int(profile.get("midi_program", 0))

            for method_name, fn in methods.items():
                out_folder = outputs_dir / inst_key / method_name / dataset_source
                out_path = out_folder / output_relative
                out_path.parent.mkdir(parents=True, exist_ok=True)

                if method_name == "patch_only":
                    patch_only_baseline(midi_path, out_path, program=program)
                    out_notes = base_notes
                    status = "UNCHANGED"
                    reason = "Patch-only baseline (no symbolic changes)"
                    shift = 0
                    tv = tessitura_violations(out_notes, profile)
                    hf = hard_feasible(out_notes, profile)
                else:
                    res = fn(base_notes, profile)
                    out_notes = res["notes"]
                    status = res["status"]
                    reason = res["reason"]
                    shift = res.get("shift", 0)

                    if out_notes is None:
                        patch_only_baseline(midi_path, out_path, program=program)
                        tv = tessitura_violations(base_notes, profile)
                        hf = False
                    else:
                        notes_to_midi(out_notes, out_path, program=program, bpm=parsed_bpm, ticks_per_beat=parsed_tpb)
                        tv = tessitura_violations(out_notes, profile)
                        hf = hard_feasible(out_notes, profile)

                out_poly = max_simultaneous_notes(out_notes) if out_notes else 0
                dropped = len(base_notes) - len(out_notes) if out_notes else len(base_notes)
                preservation = get_melody_preservation(base_notes, out_notes, shift)

                rows.append({
                    "file": str(midi_path),
                    "relative_file": str(relative_file),
                    "dataset_source": dataset_source,
                    "clip": clip_name,
                    "clip_id": clip_id,
                    "instrument": inst_key,
                    "category": profile.get("category", ""),
                    "method": method_name,
                    "status": status,
                    "hard_feasible": hf,
                    "tessitura_violations": tv,
                    "shift": shift,
                    "abs_shift": abs(shift) if shift is not None else None,
                    "input_max_polyphony": in_poly,
                    "output_max_polyphony": out_poly,
                    "notes_dropped": dropped,
                    "melody_preservation": preservation,
                    "output_midi": str(out_path),
                    "reason": reason,
                })

    return pd.DataFrame(rows)


# Demo: generate one MIDI from an existing symbolic clip, then run the file harness
demo_dir = Path("datasets/inputs/demo")
demo_dir.mkdir(exist_ok=True)
demo_in = demo_dir / "demo_input.mid"
demo_notes = EVAL_CLIPS["flute_out_of_range"]

notes_to_midi(demo_notes, demo_in, program=0, bpm=120)
print("Wrote demo input MIDI:", demo_in)

df_file = run_file_harness(demo_dir, EVAL_INSTRUMENTS, outputs_dir="datasets/outputs/demo")

# Display the dataframe, showing off the new source-aware columns!
df_file.sort_values(["dataset_source", "clip", "instrument", "method"])[
    ["dataset_source", "instrument", "method", "input_max_polyphony", "output_max_polyphony", "notes_dropped", "melody_preservation", "hard_feasible"]
].head(10)


Wrote demo input MIDI: datasets/inputs/demo/demo_input.mid


,dataset_source,instrument,method,input_max_polyphony,output_max_polyphony,notes_dropped,melody_preservation,hard_feasible
6,demo,CLARINET,constraint_aware,1,1,0,1.0,True
4,demo,CLARINET,patch_only,1,1,0,1.0,False
5,demo,CLARINET,simple_pitch,1,1,0,1.0,True
7,demo,CLARINET,texture_adapted,1,1,0,1.0,True
2,demo,FLUTE,constraint_aware,1,1,0,1.0,True
0,demo,FLUTE,patch_only,1,1,0,1.0,False
1,demo,FLUTE,simple_pitch,1,1,0,1.0,True
3,demo,FLUTE,texture_adapted,1,1,0,1.0,True
22,demo,MARIMBA,constraint_aware,1,1,0,1.0,True
20,demo,MARIMBA,patch_only,1,1,0,1.0,True


In [41]:

from pathlib import Path

if Path("datasets/inputs/uploaded_out.zip").exists() and not Path("datasets/inputs/uploaded_out").exists():
    shutil.unpack_archive("datasets/inputs/uploaded_out.zip", "datasets/inputs/uploaded_out")
    print("Unpacked datasets/inputs/uploaded_out.zip to 'datasets/inputs/uploaded_out'.")
elif Path("datasets/inputs/uploaded_out").exists():
    print("Using existing 'datasets/inputs/uploaded_out' directory.")
else:
    print("No uploaded_out.zip found under datasets/inputs; continuing without an uploaded OUT dataset.")


Unpacked datasets/inputs/uploaded_out.zip to 'datasets/inputs/uploaded_out'.


In [42]:

external_sources = {}

if Path("datasets/inputs/uploaded_out").exists():
    external_sources["uploaded_out"] = Path("datasets/inputs/uploaded_out")

if Path("datasets/inputs/free_midi_subset").exists():
    external_sources["free_midi_subset"] = Path("datasets/inputs/free_midi_subset")
elif Path("datasets/inputs/free_midi_full").exists():
    print("datasets/inputs/free_midi_subset not found; falling back to the full free-midi-chords pack.")
    external_sources["free_midi_full"] = Path("datasets/inputs/free_midi_full")

AGGREGATED_DATASET_PATH, aggregate_summary = build_aggregated_dataset(
    EVAL_CLIPS,
    aggregated_dir="datasets/inputs/aggregated",
    external_sources=external_sources,
)
AGGREGATED_OUTPUTS_DIR = Path("datasets/outputs/aggregated")

print("Aggregated dataset ready at:", AGGREGATED_DATASET_PATH)
print("Source counts:", aggregate_summary)

df_real_results = run_file_harness(
    AGGREGATED_DATASET_PATH,
    EVAL_INSTRUMENTS,
    outputs_dir=AGGREGATED_OUTPUTS_DIR,
)

df_real_results.to_csv("datasets/outputs/aggregated/phase2_aggregated_results.csv", index=False)

overall_summary = (
    df_real_results.groupby(["method"])
    .agg(
        hard_feasible_rate=("hard_feasible", "mean"),
        avg_tessitura=("tessitura_violations", "mean"),
        avg_melody_preservation=("melody_preservation", "mean"),
        avg_abs_shift=("abs_shift", "mean"),
        avg_notes_dropped=("notes_dropped", "mean"),
        avg_input_max_polyphony=("input_max_polyphony", "mean"),
        avg_output_max_polyphony=("output_max_polyphony", "mean"),
        accepted_rate=("status", lambda s: (s == "ACCEPTED").mean()),
        corrected_rate=("status", lambda s: (s == "CORRECTED").mean()),
        rejected_rate=("status", lambda s: (s == "REJECTED").mean()),
        unchanged_rate=("status", lambda s: (s == "UNCHANGED").mean()),
        n=("clip_id", "count"),
    )
    .reset_index()
    .sort_values(["method"])
)

overall_summary.to_csv("datasets/outputs/aggregated/aggregated_summary_metrics.csv", index=False)

print(f"Processing complete. Analyzed {len(df_real_results)} combinations across {df_real_results['dataset_source'].nunique()} sources.")
print("Results saved to 'datasets/outputs/aggregated/phase2_aggregated_results.csv'")
print("Summary metrics saved to 'datasets/outputs/aggregated/aggregated_summary_metrics.csv'")

source_summary = (
    df_real_results.groupby(["dataset_source", "method"])
    .agg(
        hard_feasible_rate=("hard_feasible", "mean"),
        avg_tessitura=("tessitura_violations", "mean"),
        avg_notes_dropped=("notes_dropped", "mean"),
        avg_melody_preservation=("melody_preservation", "mean"),
        n=("clip_id", "count"),
    )
    .reset_index()
    .sort_values(["dataset_source", "method"])
)

source_summary


Aggregated dataset ready at: datasets/inputs/aggregated
Source counts: {'hardcoded_eval': 8, 'uploaded_out': 39, 'free_midi_subset': 3174}
Processing complete. Analyzed 77304 combinations across 3 sources.
Results saved to 'datasets/outputs/aggregated/phase2_aggregated_results.csv'


,dataset_source,method,hard_feasible_rate,avg_tessitura,avg_notes_dropped,avg_melody_preservation,n
0,free_midi_subset,constraint_aware,0.120195,8.641567,17.450536,0.120195,19044
1,free_midi_subset,patch_only,0.001260,9.105493,0.000000,1.000000,19044
2,free_midi_subset,simple_pitch,0.120195,6.475110,9.093258,0.560912,19044
3,free_midi_subset,texture_adapted,0.953529,0.574197,13.399653,0.953529,19044
4,hardcoded_eval,constraint_aware,0.604167,0.791667,1.000000,0.604167,48
5,hardcoded_eval,patch_only,0.375000,1.416667,0.000000,1.000000,48
6,hardcoded_eval,simple_pitch,0.604167,1.020833,0.375000,0.812500,48
7,hardcoded_eval,texture_adapted,0.812500,0.666667,0.791667,0.812500,48
8,uploaded_out,constraint_aware,0.106838,852.179487,1648.995726,0.106838,234
9,uploaded_out,patch_only,0.085470,854.341880,0.000000,1.000000,234


In [43]:

import struct


def check_midi_type(file_path):
    try:
        with open(file_path, 'rb') as f:
            header = f.read(14)
            midi_format = struct.unpack('>H', header[8:10])[0]
            track_count = struct.unpack('>H', header[10:12])[0]
            print(f"File: {file_path}")
            print(f"MIDI Format: Type {midi_format}")
            print(f"Total Tracks: {track_count}")
            return midi_format, track_count
    except Exception as exc:
        print(f"Could not read file: {exc}")


preferred_names = ["Se_stiamo_insieme.1.mid", "I - C.mid"]
example_original = None
for file_name in preferred_names:
    matches = list(AGGREGATED_DATASET_PATH.rglob(file_name))
    if matches:
        example_original = matches[0]
        break

if example_original is None:
    example_original = next(AGGREGATED_DATASET_PATH.rglob("*.mid"), None)

if example_original is None:
    print("No MIDI files found in the aggregated dataset.")
else:
    check_midi_type(example_original)


File: datasets/inputs/aggregated/uploaded_out/OUT/piano_polyphony/Se_stiamo_insieme.1.mid
MIDI Format: Type 1
Total Tracks: 15


In [44]:

import shutil
from pathlib import Path

# Create the specific demo directory
chord_demo_dir = Path("ableton_chords_demo")
chord_demo_dir.mkdir(exist_ok=True)

preferred_sources = ["free_midi_subset", "free_midi_full", "uploaded_out"]
demo_triplet = pick_demo_triplet(
    df_real_results,
    AGGREGATED_DATASET_PATH,
    instrument="VIOLIN",
    preferred_clip="I - C",
    preferred_sources=preferred_sources,
)

if demo_triplet is None:
    demo_triplet = pick_demo_triplet(df_real_results, AGGREGATED_DATASET_PATH, instrument="VIOLIN")

if demo_triplet is None:
    print("Could not find a suitable violin demo triplet in the aggregated results.")
else:
    print("Using demo clip:", demo_triplet["relative_file"])

    files_to_copy = [
        (demo_triplet["original_file"], "1_Chords_ORIGINAL.mid"),
        (demo_triplet["patch_only_file"], "2_Chords_NAIVE_Violin.mid"),
        (demo_triplet["texture_adapted_file"], "3_Chords_PIPELINE_Violin.mid"),
    ]

    for src, dst_name in files_to_copy:
        if src.exists():
            shutil.copy(src, chord_demo_dir / dst_name)
            print(f"Copied: {dst_name}")
        else:
            print(f"Missing: {src} - Check if the aggregated pipeline ran successfully for this file.")

    print(f"\nDemo folder '{chord_demo_dir}' is ready for Ableton!")


Using demo clip: free_midi_subset/01 - C Major - A minor/1 Triad/Major/I - C.mid
Copied: 1_Chords_ORIGINAL.mid
Copied: 2_Chords_NAIVE_Violin.mid
Copied: 3_Chords_PIPELINE_Violin.mid

Demo folder 'ableton_chords_demo' is ready for Ableton!


In [45]:

import mido
from pathlib import Path

preferred_sources = ["free_midi_subset", "free_midi_full", "uploaded_out"]
demo_triplet = pick_demo_triplet(
    df_real_results,
    AGGREGATED_DATASET_PATH,
    instrument="VIOLIN",
    preferred_clip="I - C",
    preferred_sources=preferred_sources,
)

if demo_triplet is None:
    demo_triplet = pick_demo_triplet(df_real_results, AGGREGATED_DATASET_PATH, instrument="VIOLIN")

if demo_triplet is None:
    print("Could not find a suitable violin demo triplet in the aggregated results.")
else:
    file_paths = {
        "1. ORIGINAL": demo_triplet["original_file"],
        "2. NAIVE (Violin Baseline)": demo_triplet["patch_only_file"],
        "3. PIPELINE (Texture Adapted)": demo_triplet["texture_adapted_file"],
    }

    for label, path in file_paths.items():
        print(f"\n{'='*50}")
        print(f"{label} -> {path}")
        print(f"{'='*50}")

        mid = mido.MidiFile(path)
        note_track = None
        for track in mid.tracks:
            if any(msg.type in ['note_on', 'note_off'] for msg in track):
                note_track = track
                break

        if note_track:
            print(f"{'Type':<10} | {'Note':<4} | {'Vel':<3} | {'Delta Time'}")
            print("-" * 50)

            note_count = 0
            for msg in note_track:
                if msg.type in ['note_on', 'note_off']:
                    actual_type = "note_off" if (msg.type == 'note_on' and msg.velocity == 0) else msg.type
                    print(f"{actual_type:<10} | {msg.note:<4} | {msg.velocity:<3} | {msg.time}")
                    note_count += 1

                    if note_count >= 15:
                        print("...")
                        break
        else:
            print("No note events found in this file.")



1. ORIGINAL -> datasets/inputs/aggregated/free_midi_subset/01 - C Major - A minor/1 Triad/Major/I - C.mid
Type       | Note | Vel | Delta Time
--------------------------------------------------
note_on    | 60   | 100 | 0
note_on    | 64   | 100 | 0
note_on    | 67   | 100 | 0
note_on    | 36   | 100 | 0
note_off   | 60   | 100 | 15360
note_off   | 64   | 100 | 0
note_off   | 67   | 100 | 0
note_off   | 36   | 100 | 0

2. NAIVE (Violin Baseline) -> datasets/outputs/aggregated/VIOLIN/patch_only/free_midi_subset/01 - C Major - A minor/1 Triad/Major/I - C.mid
Type       | Note | Vel | Delta Time
--------------------------------------------------
note_on    | 60   | 100 | 0
note_on    | 64   | 100 | 0
note_on    | 67   | 100 | 0
note_on    | 36   | 100 | 0
note_off   | 60   | 100 | 15360
note_off   | 64   | 100 | 0
note_off   | 67   | 100 | 0
note_off   | 36   | 100 | 0

3. PIPELINE (Texture Adapted) -> datasets/outputs/aggregated/VIOLIN/texture_adapted/free_midi_subset/01 - C Major - A m

In [46]:

import shutil
from pathlib import Path

# Create the demo directory
success_demo_dir = Path("ableton_demo_success")
success_demo_dir.mkdir(exist_ok=True)

preferred_sources = ["free_midi_subset", "free_midi_full", "uploaded_out"]
success_triplet = pick_demo_triplet(
    df_real_results,
    AGGREGATED_DATASET_PATH,
    instrument="VIOLIN",
    preferred_sources=preferred_sources,
    min_notes_dropped=1,
)

if success_triplet is None:
    success_triplet = pick_demo_triplet(df_real_results, AGGREGATED_DATASET_PATH, instrument="VIOLIN")

if success_triplet is None:
    print("Could not find a strong texture-adaptation example in the aggregated results.")
else:
    print("Using success example:", success_triplet["relative_file"])

    files_to_copy = [
        (success_triplet["original_file"], "1_SUCCESS_ORIGINAL.mid"),
        (success_triplet["patch_only_file"], "2_SUCCESS_NAIVE_Violin.mid"),
        (success_triplet["texture_adapted_file"], "3_SUCCESS_PIPELINE_Violin.mid"),
    ]

    all_exist = True
    for src, _ in files_to_copy:
        if not src.exists():
            print(f"Missing: {src}")
            all_exist = False

    if all_exist:
        for src, dst_name in files_to_copy:
            shutil.copy(src, success_demo_dir / dst_name)
        print(f"Success! All 3 demo files are ready in '{success_demo_dir}'.")
    else:
        print("One or more files were not found. Please double-check the aggregated outputs folder.")


Using success example: uploaded_out/OUT/piano_polyphony/Luomo_Dellarmonica.4.mid
Success! All 3 demo files are ready in 'ableton_demo_success'.
